# External Framework Validation — Synthcity

**Clean, linear pipeline.** Run top to bottom (Kernel -> Restart & Run All).

This notebook computes a second, independent set of quality metrics for the
CTGAN and CTAB-GAN+ synthetic datasets on the UCI Default of Credit Card
Clients dataset, using the **Synthcity** framework (van der Schaar Lab).

It is deliberately built to mirror the SynthEval notebook (`credit_syntheval.ipynb`):
the same real reference (`real_minority_train.csv`), the same two synthetic
files (`synthetic_ctgan_final.csv`, `synthetic_ctab_final.csv`), the same
categorical/numerical column split, and the same 70/30 stratified holdout
reconstruction. This guarantees that the two frameworks are evaluated on
**identical data**, so that any agreement or disagreement between them is a
property of the metrics, not of the inputs.

The metric selection follows three principles:
1. **Direct comparison** with a SynthEval / manual metric (cross-validation).
2. **Complementary view** on a dimension already measured (different lens).
3. **Genuinely new** privacy attacks not available in SynthEval, to complement
   the manual memorisation analysis.

SMOTE is not evaluated here: in this dissertation it is used only as a
utility baseline (TSTR), not as a subject of the fidelity and privacy analysis.

**Requirements:** environment `tese_synthcity` (Python 3.10) with
`synthcity==0.2.12`, `opacus==1.4.0`, `numpy==1.26.4`.


## 0. Configuration and paths

In [19]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Synthcity is verbose on import (KeOps / transformers warnings are harmless).
warnings.filterwarnings("ignore")

from synthcity.plugins.core.dataloader import GenericDataLoader
from synthcity.metrics import Metrics

RANDOM_STATE = 123

# >>> Adjust this path if necessary <<<
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)


Data folder: data


## 1. Canonical column definitions

Defined **once** here and never reassigned, exactly as in the SynthEval
notebook. `FEATURES` is the 23 predictor columns, deliberately **excluding**
the target `def_pay`. The categorical/numerical split is identical to the one
used with SynthEval's `cat_cols=CAT_COLS`, which corresponds to **Variant A
(PAY_X treated as categorical)** — the baseline case of the DCR sensitivity analysis.


In [20]:
TARGET   = "def_pay"

FEATURES = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
            "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

MONETARY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] \
                         + [f"PAY_AMT{i}"  for i in range(1, 7)]

# Categorical/ordinal columns (matches the manual TVD treatment
# and SynthEval Variant A). PAY_1..PAY_6 are treated as categorical here.
CAT_COLS = ["SEX", "EDUCATION", "MARRIAGE"] + [f"PAY_{i}" for i in range(1, 7)]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]


assert len(FEATURES) == 23, f"FEATURES must be 23, got {len(FEATURES)}"
assert TARGET not in FEATURES, "def_pay must not be in FEATURES"
assert len(CAT_COLS) == 9 and len(NUM_COLS) == 14

print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))


FEATURES: 23 | CAT: 9 | NUM: 14


## 2. Helper: align any dataframe to the 23 features

Identical in spirit to SynthEval's `to_features`: drops `def_pay` if present,
enforces column order, and casts the monetary columns to `float64` so that
CTAB-GAN+ integer output matches the rest. Used for the fidelity block, which
(as in SynthEval) is computed on the 23 features **without** the target.


In [21]:
def to_features(df, name):
    d = df.copy()
    if TARGET in d.columns:
        d = d.drop(columns=[TARGET])
    d = d[FEATURES]                       # enforce order; errors if any missing
    d[MONETARY] = d[MONETARY].astype("float64")
    assert list(d.columns) == FEATURES
    print(f"{name:<16} -> {d.shape}")
    return d


def with_target(df, name):
    """Align to the 23 features AND keep def_pay as the last column.

    Used for the privacy block, which evaluates the full record including the label.
    """
    d = df.copy()
    assert TARGET in d.columns, f"{name}: '{TARGET}' column is required here"
    cols = FEATURES + [TARGET]
    d = d[cols]                           # enforce order; errors if any missing
    d[MONETARY] = d[MONETARY].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == cols
    print(f"{name:<16} -> {d.shape}")
    return d


## 3. Load the synthetic datasets and the real minority reference

**Exactly the same files as the SynthEval notebook.** The real reference is the
real minority partition (`real_minority_train.csv`); the synthetic sets are the
`*_final.csv` files. Two aligned versions of each are prepared: one without the
target (for fidelity) and one with the target (for privacy).


In [22]:
real_min  = pd.read_csv(DATA_DIR / "real_minority_train.csv")
synth_ctg = pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv")
synth_ctb = pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv")

# --- feature-only views (no target): for fidelity / sanity / detection ---
print("Feature-only views (no target):")
real_min_p  = to_features(real_min,  "real_minority")
synth_ctg_p = to_features(synth_ctg, "ctgan")
synth_ctb_p = to_features(synth_ctb, "ctabganplus")

# --- views with target: for privacy ---
print("\nViews with target:")
real_min_t  = with_target(real_min,  "real_minority")
synth_ctg_t = with_target(synth_ctg, "ctgan")
synth_ctb_t = with_target(synth_ctb, "ctabganplus")

# columns must be perfectly aligned within each view
for d in (synth_ctg_p, synth_ctb_p):
    assert list(d.columns) == list(real_min_p.columns)
for d in (synth_ctg_t, synth_ctb_t):
    assert list(d.columns) == list(real_min_t.columns)
print("\nAll columns aligned.")


Feature-only views (no target):
real_minority    -> (4645, 23)
ctgan            -> (11710, 23)
ctabganplus      -> (11710, 23)

Views with target:
real_minority    -> (4645, 24)
ctgan            -> (11710, 24)
ctabganplus      -> (11710, 24)

All columns aligned.


## 4. Reconstruct the canonical split → holdout

Rebuilds the 70/30 stratified split from raw data with `RANDOM_STATE = 123`,
then verifies it matches `real_train_full.csv` exactly (bit for bit), exactly
as in the SynthEval notebook. The test partition becomes the reference
**holdout** used by the privacy attacks (membership inference needs a set of
records that were *not* used to build the synthetic data).


In [23]:
df_raw = pd.read_csv(DATA_DIR / "UCI_Credit_Card.csv")

df_c = (df_raw
        .drop(columns=["ID"])
        .rename(columns={"PAY_0": "PAY_1",
                         "default.payment.next.month": TARGET}))
df_c["EDUCATION"] = df_c["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df_c["MARRIAGE"]  = df_c["MARRIAGE"].replace({0: 3})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
assert train_recon.equals(real_full.reset_index(drop=True)), \
    "Reconstructed split != real_train_full.csv"
print("Split reproduced bit for bit. OK.")

# Holdout WITH target, aligned to the same schema as the with-target views.
holdout_df = X_test.copy()
holdout_df[TARGET] = y_test
holdout_t = with_target(holdout_df, "holdout")
assert list(holdout_t.columns) == FEATURES + [TARGET]
print("Holdout ready:", holdout_t.shape)


Split reproduced bit for bit. OK.
holdout          -> (9000, 24)
Holdout ready: (9000, 24)


## 5. Wrap the data in Synthcity `GenericDataLoader`

Synthcity does not take bare DataFrames like SynthEval does; it works with a
`DataLoader` object that carries the schema (which columns are the target and
which are discrete). Building the loaders once, here, keeps the metric calls
below clean.

Two families of loaders are built:

* **Feature-only loaders** (no target) — for fidelity, sanity and detection,
  matching how SynthEval computed KS, correlation and nearest-neighbour
  distance on the 23 features alone.
* **With-target loaders** — for the privacy (identifiability) metric, which
  uses the full record including the label.

`target_column` is set only on the with-target loaders. Synthcity 0.2.12 has no argument to declare discrete columns: it treats a
column as categorical when it is non-numeric or has fewer than 15 distinct
values. This covers `SEX`, `EDUCATION`, `MARRIAGE`, `PAY_X` and the target,
the same categorical set declared in SynthEval Variant A.


In [25]:
def make_loader(df, target=None):
    """Wrap a dataframe in a GenericDataLoader.

    Synthcity 0.2.12 has no argument to declare discrete columns: its encoder
    treats a column as categorical when it is non-numeric or has fewer than
    15 distinct values.
    """
    return GenericDataLoader(
        df.reset_index(drop=True),
        target_column=target,
    )

# --- feature-only loaders (no target) ---
loader_real_feat = make_loader(real_min_p)
loader_ctg_feat  = make_loader(synth_ctg_p)
loader_ctb_feat  = make_loader(synth_ctb_p)

# --- with-target loaders ---
loader_real_tgt = make_loader(real_min_t, target=TARGET)
loader_ctg_tgt  = make_loader(synth_ctg_t, target=TARGET)
loader_ctb_tgt  = make_loader(synth_ctb_t, target=TARGET)

# holdout loader (with target) for the membership-inference reference set
loader_holdout_tgt = make_loader(holdout_t, target=TARGET)

print("Loaders built.")
print("  real (feat) :", loader_real_feat.dataframe().shape)
print("  real (tgt)  :", loader_real_tgt.dataframe().shape)
print("  holdout(tgt):", loader_holdout_tgt.dataframe().shape)


Loaders built.
  real (feat) : (4645, 23)
  real (tgt)  : (4645, 24)
  holdout(tgt): (9000, 24)


In [24]:
# Minority-only holdout for the membership-inference attack.
# DomiasMIA assumes members (X_train) and non-members (X_ref_syn) come from
# the SAME distribution. Since the generation reference is minority-only,
# the non-member set must also be minority-only, otherwise the attack would
# detect a class-distribution shift rather than genuine privacy leakage.
holdout_min = holdout_df[holdout_df[TARGET] == 1].copy()
holdout_min_t = with_target(holdout_min, "holdout_minority")
loader_holdout_min = make_loader(holdout_min_t, target=TARGET)
print("Minority holdout:", holdout_min_t.shape)

holdout_minority -> (1991, 24)
Minority holdout: (1991, 24)


## 6. Metric selection

The metrics are grouped by how they relate to the existing SynthEval / manual
results. Synthcity's `Metrics.evaluate` takes a dictionary
`{category: [metric, ...]}` using the exact names returned by `Metrics.list()`.

| Synthcity metric | category | role | relates to |
|---|---|---|---|
| `ks_test` | stats | **direct comparison** | SynthEval KS |
| `nearest_syn_neighbor_distance` | sanity | **direct comparison** | manual DCR |
| `prdc` | stats | complementary | richer view of fidelity/coverage |
| `alpha_precision` | stats | complementary | fidelity + sampling quality |
| `detection_xgb` | detection | complementary | global distinguishability |
| `identifiability_score` | privacy | **new** | complements the manual memorisation analysis |

The two blocks below split the metrics by whether they need the target.

Two Synthcity metrics are deliberately excluded: the downstream `xgb`
performance metric (the real reference is the minority class only, so a
default classifier is undefined) and the `DomiasMIA_KDE` membership-inference
attack (its kernel density estimate fails on a singular covariance matrix
caused by the collinear financial features).


In [35]:
# Fidelity / sanity / detection — computed on the feature-only views.
METRICS_FEAT = {
    "stats":     ["ks_test", "prdc", "alpha_precision"],
    "sanity":    ["nearest_syn_neighbor_distance"],
    "detection": ["detection_xgb"],
}

# Privacy only. The performance (xgb/TSTR) metric is intentionally dropped:
# the real reference here is the minority partition (a single class,
# def_pay == 1), so a downstream default classifier is not defined. Utility
# is already covered by the manual LR/RF TSTR evaluation.
METRICS_TGT = {
    "privacy": ["identifiability_score"],
}

print("Feature-level metrics:", METRICS_FEAT)
print("Target-level metrics :", METRICS_TGT)

Feature-level metrics: {'stats': ['ks_test', 'prdc', 'alpha_precision'], 'sanity': ['nearest_syn_neighbor_distance'], 'detection': ['detection_xgb']}
Target-level metrics : {'privacy': ['identifiability_score']}


## 7. Fidelity / sanity / detection metrics (no target)

`Metrics.evaluate` returns a tidy DataFrame indexed by metric name, with a
`mean` column (among others). The helper below runs one synthetic model
against the real minority reference and tags the result with the model name,
mirroring the `tag()` pattern in the SynthEval notebook.


In [27]:
def eval_block(loader_real, loader_synth, metrics_dict, model_name):
    print(f"\n{'='*55}\n{model_name}\n{'='*55}")
    res = Metrics.evaluate(
        loader_real,
        loader_synth,
        metrics=metrics_dict,
        task_type="classification",
        random_state=RANDOM_STATE,
        workspace="synthcity_workspace",
    )
    out = res.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "model", model_name)
    return out


fid_ctgan = eval_block(loader_real_feat, loader_ctg_feat, METRICS_FEAT, "CTGAN")
fid_ctab  = eval_block(loader_real_feat, loader_ctb_feat, METRICS_FEAT, "CTAB-GAN+")

fidelity_all = pd.concat([fid_ctgan, fid_ctab], ignore_index=True)
print("\nFidelity / sanity / detection (mean values):")
print(fidelity_all.pivot(index="model", columns="metric", values="mean").round(4))



CTGAN

CTAB-GAN+

Fidelity / sanity / detection (mean values):
metric     detection.detection_xgb.mean  \
model                                     
CTAB-GAN+                        0.9953   
CTGAN                            0.9996   

metric     sanity.nearest_syn_neighbor_distance.mean  \
model                                                  
CTAB-GAN+                                     0.0440   
CTGAN                                         0.0438   

metric     stats.alpha_precision.authenticity_OC  \
model                                              
CTAB-GAN+                                 0.0052   
CTGAN                                     0.1757   

metric     stats.alpha_precision.authenticity_naive  \
model                                                 
CTAB-GAN+                                    0.5974   
CTGAN                                        0.7225   

metric     stats.alpha_precision.delta_coverage_beta_OC  \
model                                            

## 8. Privacy metrics (with target)

The identifiability score is computed on the with-target views. The real
minority partition (the data behind the generation) is passed as the member
set, and the minority-only holdout as the non-member reference set.


In [28]:
def eval_block_tgt(loader_synth, model_name):
    print(f"\n{'='*55}\n{model_name}\n{'='*55}")
    res = Metrics.evaluate(
        loader_real_tgt,          # X_gt: real minority reference
        loader_synth,             # X_syn: the synthetic data
        X_train=loader_real_tgt,  # members: the real data behind the generation
        X_ref_syn=loader_holdout_min,  # non-members: unseen real minority
        metrics=METRICS_TGT,
        task_type="classification",
        random_state=RANDOM_STATE,
        workspace="synthcity_workspace",
    )
    out = res.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "model", model_name)
    return out


tgt_ctgan = eval_block_tgt(loader_ctg_tgt, "CTGAN")
tgt_ctab  = eval_block_tgt(loader_ctb_tgt, "CTAB-GAN+")

privacy_all = pd.concat([tgt_ctgan, tgt_ctab], ignore_index=True)
print("\nPrivacy (mean values):")
print(privacy_all.pivot(index="model", columns="metric", values="mean").round(4))


CTGAN

CTAB-GAN+

Privacy (mean values):
metric     privacy.identifiability_score.score  \
model                                            
CTAB-GAN+                               0.0986   
CTGAN                                   0.0282   

metric     privacy.identifiability_score.score_OC  
model                                              
CTAB-GAN+                                  0.3966  
CTGAN                                      0.2629  


## 9. Consolidate and save results

Two CSVs are written, following the naming convention of the SynthEval outputs
(`syntheval_fidelity_results.csv`, `syntheval_privacy_results.csv`):

* `synthcity_fidelity_results.csv`
* `synthcity_privacy_results.csv`

plus a single combined file for convenience.


In [29]:
fidelity_all.to_csv(DATA_DIR / "synthcity_fidelity_results.csv", index=False)
print("Saved: synthcity_fidelity_results.csv")

privacy_all.to_csv(DATA_DIR / "synthcity_privacy_results.csv", index=False)
print("Saved: synthcity_privacy_results.csv")

combined = pd.concat([fidelity_all, privacy_all], ignore_index=True)
combined.to_csv(DATA_DIR / "synthcity_all_results.csv", index=False)
print("Saved: synthcity_all_results.csv")


Saved: synthcity_fidelity_results.csv
Saved: synthcity_privacy_results.csv
Saved: synthcity_all_results.csv


---
**Done.** If every cell ran without error, the three result CSVs are written
and the in-memory objects (`fidelity_all`, `privacy_all`, `combined`, and the
loaders) are available for any further analysis or plotting

In [30]:
print("=== FIDELITY / SANITY / DETECTION ===")
print(fidelity_all.pivot(index="model", columns="metric", values="mean").round(4))

print("\n=== PRIVACY ===")
print(privacy_all.pivot(index="model", columns="metric", values="mean").round(4))


=== FIDELITY / SANITY / DETECTION ===
metric     detection.detection_xgb.mean  \
model                                     
CTAB-GAN+                        0.9953   
CTGAN                            0.9996   

metric     sanity.nearest_syn_neighbor_distance.mean  \
model                                                  
CTAB-GAN+                                     0.0440   
CTGAN                                         0.0438   

metric     stats.alpha_precision.authenticity_OC  \
model                                              
CTAB-GAN+                                 0.0052   
CTGAN                                     0.1757   

metric     stats.alpha_precision.authenticity_naive  \
model                                                 
CTAB-GAN+                                    0.5974   
CTGAN                                        0.7225   

metric     stats.alpha_precision.delta_coverage_beta_OC  \
model                                                     
CTAB-GAN+       